## Config

In [1]:
import os, json, re, math
from pathlib import Path
from time import time
from tqdm import tqdm

import torch
import torch.nn as nn
import tokenizers
from datasets import load_dataset, load_from_disk

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"*** Using device: {device.type} ***")

colab = False

if colab:
  from google.colab import drive
  drive.mount('/content/drive')
  ROOT_PATH = Path("/content/drive/MyDrive/ML-Projects/CausalLSTM")
else:
  ROOT_PATH = Path.cwd()

paths = {
        "DATASET_PROCESSED": ROOT_PATH / "data/wikitext2_processed",
        "DATASET_WT2": ROOT_PATH / "data/wikitext-2",
        "CONFIG": ROOT_PATH / "config",
        "TOKENIZER": ROOT_PATH / "tokenizers",
        "MODELS": ROOT_PATH / "models",
    }

for key, path in paths.items():
        path.mkdir(parents=True, exist_ok=True)

SEED = 42

*** Using device: cuda ***


In [2]:
### WRITE CONFIG FILE ###

config = {
    "vocab_size": 30_000,
    "seq_len": 70,
    "batch_size": 128,
    "n_epochs": 20,
    "enable_mixed_precision": True if device.type == "cuda" else False,
    "grad_clip_norm": 0.25,
    "early_stopping_patience": 3,
    "early_stopping_epsilon": 3e-4,
    "model_params": {
        "embedding_dim": 400,
        "hidden_dim": 1024,
        "num_layers": 2,
        "lstm_dropout_p": 0.35,
        "emb_dropout_p": 0.2,
        "out_dropout_p": 0.4,
    },
    "optimizer_params": {
        "lr": 2e-3,
        "weight_decay": 2e-5,
    },
    "lr_scheduler_params": {
        "factor": 0.5,
        "patience": 2,
        "threshold": 2e-3
    }
}

with open(paths["CONFIG"] / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f)
print(f"*** config saved to {paths["CONFIG"]} ***")

*** config saved to /mnt/c/Users/ASUS/Documents/Machine-Learning/Projects/CausalLSTM/config ***


## Tokenization

In [3]:
import re, unicodedata

def is_valid(row):
    text = row["text"].strip()
    if text == "":
        return False
    if re.match(r"^=+.*=+$", text):
        return False
    return True

def clean_text(row):
    t = row["text"]
    # lowercase + unicode → ascii
    t = unicodedata.normalize("NFKD", t.lower()).encode("ascii", "ignore").decode("ascii")
    # fix wikitext artifacts
    t = re.sub(r"@-@", "-", t)
    t = re.sub(r"@\.@", ".", t)
    t = re.sub(r"@,@", ",", t)
    t = re.sub(r"[–—−]", "-", t)
    # remove parentheses and content
    t = re.sub(r"\([^)]*\)", " ", t)
    # roman numerals i–v → digits
    t = re.sub(r"\b(i|ii|iii|iv|v)\b",lambda m: str({"i": 1, "ii": 2, "iii": 3, "iv": 4, "v": 5}[m.group()]),t,)
    # normalize ellipses (3+ dots) → "."
    t = re.sub(r"\.{3,}", ".", t)
    # keep: . , ; : ! ? ' - and spaces
    t = re.sub(r"[^a-z0-9\.\,\;\:\!\?\'\-\s]", " ", t)
    # collapse patterns like "6 , 000" or "3 , 000" → "<num>"
    t = re.sub(r"\b\d+\s*,\s*\d+\b", "<num>", t)
    # (optional) handle ordinals like 19th, 21st → "19 th"
    t = re.sub(r"\b(\d+)(st|nd|rd|th)\b", r"\1 th", t)
    # number bucketing
    def num_bucket(m):
        n = int(m.group())
        if 0 <= n <= 9:
            return str(n)
        if 10 <= n < 100:
            return "<2dnum>"
        if 1000 <= n <= 2099:
            return "<year>"
        return "<num>"
    t = re.sub(r"\d+", num_bucket, t)
    # collapse whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return {"clean_text": t}

In [4]:
### LOAD TOKENIZER TRAINING DATA ###

if colab:
    wikitext_2 = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
else:
    wikitext_2 = load_dataset(str(paths["DATASET_WT2"]))

wikitext_2 = wikitext_2.filter(is_valid)
print(f"** total wiki-2 train set rows: {wikitext_2["train"].num_rows:,}")
wikitext_2 = wikitext_2.map(clean_text)

** total wiki-2 train set rows: 17,556


In [5]:
### TRAIN & SAVE TOKENIZER ###
from tqdm import tqdm
from tokenizers import Tokenizer, Regex
from tokenizers.models import BPE, WordLevel
from tokenizers.trainers import BpeTrainer, WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFKD, Lowercase, Replace, Sequence, StripAccents
from tokenizers.processors import TemplateProcessing

def tokenizer_trainer(
    training_iterator,
    vocab_size: int,
    save_path: str,
    special_tokens: list = ["<pad>", "<unk>", "<eos>", "<num>", "<year>", "<2dnum>"]
):
    tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    tokenizer.normalizer = Lowercase()
    # define special tokens template
    tokenizer.post_processor = TemplateProcessing(
        single="$0 <eos>",
        pair="$A <eos> $B:1 <eos>:1",
        special_tokens=[("<eos>", 2)],
    )

    trainer = WordLevelTrainer(
        vocab_size = vocab_size,
        special_tokens = special_tokens,
        show_progress = False
    )

    tokenizer.train_from_iterator(training_iterator, trainer)
    print(f"*** vocab size: {tokenizer.get_vocab_size():,} ***")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    tokenizer.save(save_path)
    print(f"*** trained tokenizer saved to: {save_path} ***")

tokenizer_trainer(tqdm(wikitext_2["train"]["clean_text"]), config["vocab_size"], str(paths["TOKENIZER"] / "tokenizer.json"))

100%|██████████████████████████████████████████████████████████████████████████| 17556/17556 [00:00<00:00, 39292.41it/s]


*** vocab size: 30,000 ***
*** trained tokenizer saved to: /mnt/c/Users/ASUS/Documents/Machine-Learning/Projects/CausalLSTM/tokenizers/tokenizer.json ***


In [6]:
### PREPARE & TOKENIZE MAIN DATASET ###
tokenizer = tokenizers.Tokenizer.from_file(str(paths["TOKENIZER"] / "tokenizer.json"))

text = wikitext_2["train"]["clean_text"][4101]
tokens = tokenizer.encode(text, add_special_tokens=True).tokens
print(text)
print("="*89)
print(tokens)
print("="*89)
print(f"count of tokens: {len(tokens)}")

crazy in love has various remixes , including the rockwilder remix , maurice 's nu soul remix , and juniors world remix . these versions appeared on the single releases of crazy in love under an alternative spelling , krazy in luv . the rockwilder remix slows down the beat and makes the song deeper and funkier with chopped up horn samples and sparkling synth textures . maurice 's nu soul remix speeds up the beat , taking it from hip - hop to house territory . a version of the song included on asian releases of dangerously in love features a rap in mandarin chinese performed by american - taiwanese singer vanness wu , instead of jay z 's performance .
['crazy', 'in', 'love', 'has', 'various', 'remixes', ',', 'including', 'the', 'rockwilder', 'remix', ',', 'maurice', "'", 's', '<unk>', 'soul', 'remix', ',', 'and', 'juniors', 'world', 'remix', '.', 'these', 'versions', 'appeared', 'on', 'the', 'single', 'releases', 'of', 'crazy', 'in', 'love', 'under', 'an', 'alternative', 'spelling', ','

In [7]:
def tokenize_fn(batch):
    encodings = tokenizer.encode_batch(batch["clean_text"], add_special_tokens=True)
    return {"token_ids": [enc.ids for enc in encodings]}

wikitext_2 = wikitext_2.map(tokenize_fn, batched=True)

print(f"*** count of tokens (whitespace splitted): {len(" ".join(wikitext_2["train"]["clean_text"]).split()):,} ***")
print(f"*** count of tokens (tokenizer splitted): {sum(len(ids) for ids in wikitext_2["train"]["token_ids"]):,} ***")
wikitext_2.save_to_disk(str(paths["DATASET_PROCESSED"]))
print(f"*** tokenized dataset saved to: {str(paths["DATASET_PROCESSED"])} ***")

*** count of tokens (whitespace splitted): 1,893,749 ***
*** count of tokens (tokenizer splitted): 1,934,894 ***


Saving the dataset (0/1 shards):   0%|          | 0/17556 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1841 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2185 [00:00<?, ? examples/s]

*** tokenized dataset saved to: /mnt/c/Users/ASUS/Documents/Machine-Learning/Projects/CausalLSTM/data/wikitext2_processed ***


## Stream Dataset

In [8]:
%%writefile src/dataset.py
import torch
from torch.utils.data import Dataset

class TruncatedBPTTDataset(Dataset):
    """
    Fixed-stream dataset for truncated BPTT. (github.com/HooM4N/CausalLSTM)
    Splits a long token sequence into batch-aligned streams and
    returns (x, y) pairs of length `seq_len` for next-token prediction. 
    """
    def __init__(self, corpus_tokens: list, batch_size: int =256, seq_len: int = 128):
        full_seq = torch.tensor(corpus_tokens, dtype=torch.long)
        # trim to multiple of batch_size
        stream_len = full_seq.size(0) // batch_size
        full_seq = full_seq[:stream_len * batch_size]
        full_seq = full_seq.view(batch_size, stream_len)

        self.seq_len = seq_len
        self.batch_size = batch_size
        self.full_seq = full_seq
        self.stream_len = full_seq.size(1) - 1

    def __len__(self):
        """Returns number of batches"""
        return self.stream_len // self.seq_len

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        x = self.full_seq[:, start:end]
        y = self.full_seq[:, start+1:end+1]
        return x, y

Overwriting src/dataset.py


## Model

In [2]:
%%writefile model.py
from typing import Tuple
import torch
import torch.nn as nn

class CausalLSTM(nn.Module):
    """ 
    LSTM-based next-token predictor with tied input/output embeddings. 
    Supports optional hidden→embedding projection when dims differ, plus
    dropout on embeddings, LSTM outputs, and projection outputs. 
    """
    def __init__(
        self, vocab_size: int, embedding_dim: int=512, hidden_dim:int=1024, num_layers:int=2,
        lstm_dropout_p:float=0.4, emb_dropout_p:float=0.25, out_dropout_p:float=0.5
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.emb_dropout = nn.Dropout(emb_dropout_p)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers, batch_first=True, dropout = lstm_dropout_p
        )
        self.out_dropout = nn.Dropout(out_dropout_p)
        self.use_proj = True if self.hidden_dim != self.embedding_dim else False
        if self.use_proj:
            self.proj = nn.Linear(hidden_dim, embedding_dim)
        self.fc = nn.Linear(embedding_dim, vocab_size)
        self.fc.weight = self.embedding.weight
        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        if self.use_proj:
            nn.init.uniform_(self.proj.weight, -initrange, initrange)
            nn.init.zeros_(self.proj.bias)
        nn.init.uniform_(self.embedding.weight, -initrange, initrange)
        nn.init.zeros_(self.fc.bias)

    def init_hidden(self, batch_size):
        w = next(self.parameters())
        h_0 = w.new_zeros((self.num_layers, batch_size, self.hidden_dim))
        c_0 = w.new_zeros((self.num_layers, batch_size, self.hidden_dim))
        return (h_0, c_0)

    def forward(self, x, hidden):
        x = self.embedding(x) # (N, L, E)
        x = self.emb_dropout(x)
        x, hidden = self.lstm(x, hidden) # (N, L, H), ((num_layers, N, H), (num_layers, N, H))
        x = self.out_dropout(x)
        if self.use_proj:
            x = self.proj(x) # (N, L, E)
            x = self.out_dropout(x)
        return self.fc(x).permute(0, 2, 1), hidden # (N, vocab_size, L), ((num_layers, N, H), (num_layers, N, H))

def detach_hidden(hidden):
    "detaches hidden from current graph"
    if isinstance(hidden, torch.Tensor):
        return hidden.detach()
    else:
        return tuple(detach_hidden(h) for h in hidden)

Overwriting model.py


## Training

In [10]:
### LOAD & PREPARE DATASET ###
from src.dataset import StreamLMDataset

ds = load_from_disk(str(paths["DATASET_PROCESSED"]))
print(f"*** total count of training tokens: {sum(len(i) for i in ds["train"]["token_ids"]):,} ***")
print(ds)

train_ds = StreamLMDataset(ds["train"]["token_ids"], config["batch_size"], config["seq_len"])
val_ds = StreamLMDataset(ds["validation"]["token_ids"], config["batch_size"], config["seq_len"])

print(f"*** batch size: {train_ds.batch_size} | sequence lenght: {train_ds.seq_len} ***")
print(f"*** count of training batches: {len(train_ds)} | validation batches: {len(val_ds)} ***")
print(f"*** stream lenght of train set: {train_ds.stream_len} | validation set: {val_ds.stream_len} ***")

*** total count of training tokens: 1,934,894 ***
DatasetDict({
    train: Dataset({
        features: ['text', 'clean_text', 'token_ids'],
        num_rows: 17556
    })
    validation: Dataset({
        features: ['text', 'clean_text', 'token_ids'],
        num_rows: 1841
    })
    test: Dataset({
        features: ['text', 'clean_text', 'token_ids'],
        num_rows: 2185
    })
})
*** batch size: 128 | sequence lenght: 70 ***
*** count of training batches: 215 | validation batches: 22 ***
*** stream lenght of train set: 15115 | validation set: 1586 ***


In [11]:
### LOAD & PREPARE MODEL ###
from src.model import CausalLSTM, detach_hidden

torch.manual_seed(SEED)
model = CausalLSTM(config["vocab_size"], **config["model_params"]).to(device)
print(f"*** total count of trainable parameters: {sum([p.numel() for p in model.parameters() if p.requires_grad]):,} ***")
print(model)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters(), **config["optimizer_params"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, **config["lr_scheduler_params"])

*** total count of trainable parameters: 26,677,696 ***
CausalLSTM(
  (embedding): Embedding(30000, 400)
  (emb_dropout): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(400, 1024, num_layers=2, batch_first=True, dropout=0.35)
  (out_dropout): Dropout(p=0.4, inplace=False)
  (proj): Linear(in_features=1024, out_features=400, bias=True)
  (fc): Linear(in_features=400, out_features=30000, bias=True)
)


In [12]:
@torch.no_grad()
def evaluate(eval_dataset, disable_progress_bar=True):
    model.eval()
    total_loss = 0.0
    hidden = model.init_hidden(config["batch_size"])
    for idx in tqdm(range(len(eval_dataset)), disable = disable_progress_bar):
        X, Y = eval_dataset[idx]
        X, Y = X.to(device), Y.to(device)
        hidden = detach_hidden(hidden)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=config["enable_mixed_precision"]):
            logits, hidden = model(X, hidden)
            total_loss += loss_fn(logits, Y).item()
    return total_loss / len(eval_dataset)

In [13]:
def train(saved_checkpoint_path = None):
    if saved_checkpoint_path is not None:
        model.load_state_dict(
            torch.load(saved_checkpoint_path, map_location=device, weights_only=True)
            )
    train_logs = {"train_loss":[] , "val_loss":[] , "val_metric":[], "lr":[]}
    model.train()
    scaler = torch.amp.GradScaler(enabled = config["enable_mixed_precision"])
    best_loss, es_counter = float('inf'), 0

    for epoch in range(config["n_epochs"]):
        start_time = time()
        model.train()
        total_loss = 0.0
        hidden = model.init_hidden(config["batch_size"])

        for idx in tqdm(range(len(train_ds)), desc=f"Epoch {epoch+1}/{config["n_epochs"]}"):
            X, Y = train_ds[idx]
            X, Y = X.to(device), Y.to(device)
            optimizer.zero_grad(set_to_none=True)
            # detach hidden from current graph
            hidden = detach_hidden(hidden)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled = config["enable_mixed_precision"]):
                logits, hidden = model(X, hidden)
                loss = loss_fn(logits, Y)
            total_loss += loss.item()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            # clip gradients to avoid exloding
            nn.utils.clip_grad_norm_(model.parameters(), config["grad_clip_norm"])
            scaler.step(optimizer)
            scaler.update()

        # logger
        train_logs["train_loss"].append(total_loss / len(train_ds))
        val_loss = evaluate(val_ds)
        train_logs["val_loss"].append(val_loss)
        train_logs["val_metric"].append(math.exp(val_loss))
        train_logs["lr"].append(optimizer.param_groups[0]['lr'])

        print(f"\r Epoch {epoch + 1}/{config["n_epochs"]}", end="")
        print(f", train loss: {train_logs["train_loss"][-1]:.4f}", end="")
        print(f', val loss: {train_logs["val_loss"][-1]:.4f}', end="")
        print(f', val perplexity: {train_logs["val_metric"][-1]:.4f}', end="")
        print(f", lr: {train_logs["lr"][-1]}", end="")
        print(f', epoch time: {time() - start_time:.2f}s')

        # learning rate scheduler
        scheduler.step(val_loss)
        torch.save(model.state_dict(), paths["MODELS"] / f"CausualLSTM_ckpnt_{epoch+1}.pt")

        # early stopping
        diff = best_loss - val_loss
        if diff >= config["early_stopping_epsilon"]:
            best_loss = val_loss
            es_counter = 0
        else:
            es_counter += 1
        if es_counter >= config["early_stopping_patience"]:
            print(f"*** early stopping triggered at epoch: {epoch+1} ***")
            break

    return model, train_logs

In [14]:
ckpnt = str(paths["MODELS"] / f"CausualLSTM_ckpnt_{13}.pt")
model, train_logs = train()

with open(os.path.join(ROOT_PATH, "training_logs.json"), "w") as f:
    json.dump(train_logs, f)

Epoch 1/20: 100%|█████████████████████████████████████████████████████████████████████| 215/215 [01:06<00:00,  3.25it/s]


 Epoch 1/20, train loss: 7.0531, val loss: 8.7637, val perplexity: 6397.8865, lr: 0.002, epoch time: 68.97s


Epoch 2/20: 100%|█████████████████████████████████████████████████████████████████████| 215/215 [01:07<00:00,  3.17it/s]


 Epoch 2/20, train loss: 5.8124, val loss: 5.4411, val perplexity: 230.7034, lr: 0.002, epoch time: 70.70s


Epoch 3/20: 100%|█████████████████████████████████████████████████████████████████████| 215/215 [01:05<00:00,  3.28it/s]


...

 Epoch 18/20, train loss: 4.5107, val loss: 4.6944, val perplexity: 109.3357, lr: 0.002, epoch time: 68.40s


Epoch 19/20: 100%|████████████████████████████████████████████████████████████████████| 215/215 [01:08<00:00,  3.16it/s]


 Epoch 19/20, train loss: 4.4974, val loss: 4.7044, val perplexity: 110.4343, lr: 0.002, epoch time: 70.97s


Epoch 20/20: 100%|████████████████████████████████████████████████████████████████████| 215/215 [01:06<00:00,  3.21it/s]


 Epoch 20/20, train loss: 4.4840, val loss: 4.6975, val perplexity: 109.6763, lr: 0.002, epoch time: 69.86s


## Inference

In [1]:
import tokenizers, json, torch
from src.model import CausalLSTM

checkpoint_path = "models/CausualLSTM_ckpnt_18.pt"
config_path = "config/config.json"
tokenizer_path = "tokenizers/tokenizer.json"

with open(config_path, "r") as f:
    config = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tokenizers.Tokenizer.from_file(tokenizer_path)
model = CausalLSTM(config["vocab_size"], **config["model_params"]).to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))

<All keys matched successfully>

In [4]:
def generate(model, tokenizer, config, device, init_word, max_new_tokens=50, temperature=1.0, seed=42):
    torch.manual_seed(seed)
    model.eval()
    hidden = model.init_hidden(1)
    unk_id, eos_id = tokenizer.token_to_id("<unk>"), tokenizer.token_to_id("<eos>")
    init_idx = tokenizer.token_to_id(init_word)
    if init_idx is not None:
        input_ = torch.tensor(init_idx, dtype=torch.long).reshape(1,-1).to(device)
    else:
        raise Exception(f"{init_word} is not in vocab")
        
    generated_words = [init_word]
    with torch.no_grad():
        for _ in range(max_new_tokens):
            output, hidden = model(input_, hidden)
            probs = output.squeeze().div(temperature).exp().cpu()
            while True:
                token_idx = torch.multinomial(probs, 1)[0]
                if token_idx != unk_id: break
            input_.fill_(token_idx)
            generated_words.append(tokenizer.id_to_token(token_idx) if token_idx != eos_id else "\n")
    return " ".join(generated_words)

In [39]:
print(generate(model, tokenizer, config, device, init_word = "we", max_new_tokens=50, temperature=1.0, seed=1))

we reach the title than they could spend less months . seeing them like the staples kingship , they were impure , who put it onto the price outside the ground for their present world . several days later , campaigners reported that the slump used in both catapults and those
